# 03 · Ablasyon (Ö1, Ö5) ve hız ölçümü (Ö9)

01 notebook'unda eğitilen ablasyon modellerini, makaledeki modelle **aynı protokolde** karşılaştırır:
Avenue kare düzeyi AUC, Avenue → UCF-Crime, Avenue → UR Fall (ADL–düşme ve aynı video içi), normal veride yeniden yapılandırma hatası.
Son bölümde makaledeki modelin T4 GPU ve CPU'daki hızı ölçülür.

**Önce** 01 notebook'unu şu `RUN` değerleriyle çalıştırın: `'abl_skip_yok'`, `'abl_skip_derin'`, `'abl_artirma_yok'`, `'abl_pencere16'`.
Henüz eğitilmemiş modeller atlanır; hepsi bitince bu notebook'u tekrar çalıştırın. 02 notebook'u bir kez çalışmış olmalı (önbellekleri kullanır).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, glob, random, warnings, time, shutil, zipfile, tarfile, urllib.request
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn

from scipy.io import loadmat
from sklearn.metrics import roc_auc_score

In [ ]:
class CFG:
    PROJ       = '/content/drive/MyDrive/AnomalyProject'
    BASE_DIR   = f'{PROJ}/frames'
    TRAIN_DIR  = f'{BASE_DIR}/avenue_train'
    AVENUE_RAW = f'{PROJ}/data/Avenue_Dataset'
    AVENUE_GT  = f'{PROJ}/avenue_gt/testing_label_mask'
    CROSS = {'Normal': (f'{BASE_DIR}/ucf_normal', 0), 'Kavga (Fighting)': (f'{BASE_DIR}/ucf_fighting', 1),
             'Hırsızlık (Stealing)': (f'{BASE_DIR}/ucf_robbery', 1), 'Soygun (Burglary)': (f'{BASE_DIR}/ucf_accident', 1)}
    MODELS = {   # ad: (model dosyası, açıklama)
        'Tam (makaledeki model)': (f'{PROJ}/model_90plus.pth',          'Atlama bağlantısı 3 seviyede, veri artırma var, pencere 8'),
        'Atlama bağlantısı yok':  (f'{PROJ}/model_abl_skip_yok.pth',    'Hiçbir seviyede atlama bağlantısı yok'),
        'Yalnızca derin seviye':  (f'{PROJ}/model_abl_skip_derin.pth',  'Yalnızca E3 ↔ D3 atlama bağlantısı'),
        'Veri artırma yok':       (f'{PROJ}/model_abl_artirma_yok.pth', 'Çevirme, afin dönüşüm ve bulanıklaştırma kapalı'),
        'Pencere 16 kare':        (f'{PROJ}/model_abl_pencere16.pth',   'Giriş 16 kare'),
    }
    URF_BASE  = 'http://fenix.ur.edu.pl/~mkepski/ds/data'
    URF_CACHE = f'{PROJ}/data/urfall_cache'
    N_ADL, N_FALL = 40, 30
    CACHE_DIR = f'{PROJ}/revizyon_sonuclari/cache'
    OUT_DIR   = f'{PROJ}/revizyon_sonuclari_v2/ablasyon'
    TMP       = '/content/tmp'
    IMG_SIZE=64; SEQ_LEN=8; SEQ_STEP=2; BATCH=64; VAL_FRAC=0.10; VAL_SPLIT_SEED=42
    AV_STRIDE = 2       # 02 notebook'unda doğrulandı: eğitim kareleri her 2. kare
    URF_STRIDE = 1
    SEED = 42

for d in [CFG.OUT_DIR, CFG.TMP, CFG.URF_CACHE]: os.makedirs(d, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz:', DEVICE)
for k, (p, _) in CFG.MODELS.items(): print(f'  {"✅" if os.path.exists(p) else "—"} {k}')

In [ ]:
EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]

def list_frames(folder, natural=True):
    fs = [f for f in os.listdir(folder) if f.lower().endswith(EXTS)]
    fs = sorted(fs, key=natural_key) if natural else sorted(fs)
    return [os.path.join(folder, f) for f in fs]

def to_gray64(img):
    # Ana notebook'taki load_frame ile aynı: gri ton + 64x64 BILINEAR, [0,1]
    if not isinstance(img, Image.Image): img = Image.fromarray(img)
    return np.asarray(img.convert('L').resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.BILINEAR),
                      np.float32) / 255.0

def load_stack(paths, cache_name=None, cache_dir=None):
    cache_dir = cache_dir or CFG.CACHE_DIR
    if cache_name:
        cp = os.path.join(cache_dir, cache_name + '.npy')
        if os.path.exists(cp):
            arr = np.load(cp)
            if len(arr) == len(paths): return arr
    arr = np.stack([to_gray64(Image.open(p)) for p in paths])
    if cache_name:
        os.makedirs(cache_dir, exist_ok=True); np.save(os.path.join(cache_dir, cache_name + '.npy'), arr)
    return arr

def cached_array(name, fn, cache_dir=None):
    cp = os.path.join(cache_dir or CFG.CACHE_DIR, name + '.npy')
    if os.path.exists(cp): return np.load(cp)
    arr = fn(); os.makedirs(os.path.dirname(cp), exist_ok=True); np.save(cp, arr); return arr

def frames_from_avi(path):
    import cv2
    cap, out = cv2.VideoCapture(path), []
    while True:
        ok, f = cap.read()
        if not ok: break
        out.append(to_gray64(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    return np.stack(out)

def fetch(url, dst, tries=3):
    # Dosyayı indirir; zaten varsa atlar. Başarısızsa False döner.
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for k in range(tries):
        try:
            urllib.request.urlretrieve(url, dst + '.part'); os.replace(dst + '.part', dst); return True
        except Exception as e:
            print(f'   indirme hatası ({k+1}/{tries}) {os.path.basename(url)}: {e}')
            time.sleep(3)
    return False

# ── UCSD Ped2 ─────────────────────────────────────────────────────────
def parse_ped_gt(m_path):
    txt = open(m_path, encoding='latin-1').read()
    out = []
    for body in re.findall(r'gt_frame\s*=\s*\[([^\]]*)\]', txt):
        fr = []
        for a, b in re.findall(r'(\d+)\s*:\s*(\d+)', body): fr += list(range(int(a), int(b) + 1))
        rest = re.sub(r'\d+\s*:\s*\d+', ' ', body)
        fr += [int(x) for x in re.findall(r'\d+', rest)]
        out.append(sorted(set(fr)))
    return out

def prepare_ped2():
    cache = CFG.PED2_CACHE
    if os.path.exists(f'{cache}/done.json'):
        meta = json.load(open(f'{cache}/done.json'))
        tr = [np.load(f'{cache}/train_{i:02d}.npy') for i in range(meta['n_train'])]
        te = [np.load(f'{cache}/test_{i:02d}.npy') for i in range(meta['n_test'])]
        lb = [np.load(f'{cache}/label_{i:02d}.npy') for i in range(meta['n_test'])]
        return tr, te, lb
    tgz = f'{CFG.TMP}/UCSD_Anomaly_Dataset.tar.gz'
    print('UCSD veri seti indiriliyor (~700 MB)...')
    if not fetch(CFG.PED2_URL, tgz): raise RuntimeError('Ped2 indirilemedi: ' + CFG.PED2_URL)
    ex = f'{CFG.TMP}/ucsd'
    if not os.path.exists(ex):
        with tarfile.open(tgz) as t: t.extractall(ex)
    root = next(r for r, ds, _ in os.walk(ex) if os.path.basename(r) == 'UCSDped2')
    trd = sorted([d for d in glob.glob(f'{root}/Train/*') if re.fullmatch(r'Train\d+', os.path.basename(d))], key=natural_key)
    ted = sorted([d for d in glob.glob(f'{root}/Test/*') if re.fullmatch(r'Test\d+', os.path.basename(d))], key=natural_key)
    gt  = parse_ped_gt(glob.glob(f'{root}/Test/*.m')[0])
    os.makedirs(cache, exist_ok=True)
    tr, te, lb = [], [], []
    for i, d in enumerate(trd):
        a = load_stack(list_frames(d)); np.save(f'{cache}/train_{i:02d}.npy', a); tr.append(a)
    for i, d in enumerate(ted):
        a = load_stack(list_frames(d)); l = np.zeros(len(a), int)
        for f in gt[i]:
            if 1 <= f <= len(a): l[f - 1] = 1
        np.save(f'{cache}/test_{i:02d}.npy', a); np.save(f'{cache}/label_{i:02d}.npy', l); te.append(a); lb.append(l)
    json.dump({'n_train': len(tr), 'n_test': len(te)}, open(f'{cache}/done.json', 'w'))
    print(f'Ped2 hazır: {len(tr)} eğitim, {len(te)} test videosu')
    return tr, te, lb

In [ ]:
class CB3(nn.Module):
    def __init__(self, ic, oc, k=(3,3,3), s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv3d(ic, oc, k, s, p, bias=False),
                                 nn.BatchNorm3d(oc), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class AE3D(nn.Module):
    # skip = (E1, E2, E3) seviyelerinde atlama bağlantısı var mı. Varsayılan: hepsi (makaledeki model).
    def __init__(self, skip=(True, True, True)):
        super().__init__()
        self.skip = tuple(skip); s1, s2, s3 = self.skip
        self.e1=nn.Sequential(CB3(1,32),CB3(32,32));     self.p1=nn.MaxPool3d((1,2,2),(1,2,2))
        self.e2=nn.Sequential(CB3(32,64),CB3(64,64));    self.p2=nn.MaxPool3d(2,2)
        self.e3=nn.Sequential(CB3(64,128),CB3(128,128)); self.p3=nn.MaxPool3d(2,2)
        self.bn=nn.Sequential(CB3(128,128),CB3(128,64),CB3(64,128),nn.Dropout3d(0.4))
        self.u3=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d3=nn.Sequential(CB3(256 if s3 else 128,128),CB3(128,64))
        self.u2=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d2=nn.Sequential(CB3(128 if s2 else 64,64),CB3(64,32))
        self.u1=nn.Upsample(scale_factor=(1,2,2),mode='trilinear',align_corners=False)
        self.d1=nn.Sequential(CB3(64 if s1 else 32,32),CB3(32,32))
        self.out=nn.Sequential(nn.Conv3d(32,1,1),nn.Sigmoid())
        for m in self.modules():
            if isinstance(m, nn.Conv3d): nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
            elif isinstance(m, nn.BatchNorm3d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        e1=self.e1(x); e1p=self.p1(e1)
        e2=self.e2(e1p); e2p=self.p2(e2)
        e3=self.e3(e2p); e3p=self.p3(e3)
        b=self.bn(e3p)
        s1, s2, s3 = self.skip
        d=self.u3(b);  d=self.d3(torch.cat([d,e3],1) if s3 else d)
        d=self.u2(d);  d=self.d2(torch.cat([d,e2],1) if s2 else d)
        d=self.u1(d);  d=self.d1(torch.cat([d,e1],1) if s1 else d)
        return self.out(d)

def win_idx(n, step, stride=1):
    # 8 karelik pencereler: başlangıç adımı `step`, pencere içi kare aralığı `stride`
    span = (CFG.SEQ_LEN - 1) * stride + 1
    return [[s + t * stride for t in range(CFG.SEQ_LEN)] for s in range(0, n - span + 1, step)]

def model_meta(path):
    mp = path.replace('.pth', '.json')
    m = json.load(open(mp)) if os.path.exists(mp) else {}
    return tuple(bool(x) for x in m.get('skip', (True, True, True))), int(m.get('seq_len', 8))

def load_model(path):
    skip, L = model_meta(path)
    m = AE3D(skip=skip).to(DEVICE); m.load_state_dict(torch.load(path, map_location=DEVICE)); m.eval()
    return m, skip, L

@torch.no_grad()
def score_windows(model, frames, idx):
    out, ft = [], torch.from_numpy(np.ascontiguousarray(frames))
    for i in range(0, len(idx), CFG.BATCH):
        x = ft[torch.as_tensor(np.asarray(idx[i:i+CFG.BATCH]))].unsqueeze(1).to(DEVICE)
        out.append(((model(x) - x) ** 2).mean(dim=(1,2,3,4)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

def frame_level(frames, stride, scorer):
    n = len(frames); span = (CFG.SEQ_LEN - 1) * stride + 1
    if n < span: frames = np.concatenate([frames, np.repeat(frames[-1:], span - n, 0)])
    idx = win_idx(len(frames), 1, stride); ws = scorer(frames, idx)
    acc = np.zeros(len(frames)); cnt = np.zeros(len(frames))
    for w, s in zip(idx, ws): acc[w] += s; cnt[w] += 1
    return (acc / np.maximum(cnt, 1))[:n]

def auc(y, s):
    y = np.asarray(y); return float(roc_auc_score(y, s)) if 0 < y.mean() < 1 else float('nan')

## Veri (02 notebook'unun önbelleklerinden)

In [ ]:
mats = sorted(glob.glob(f'{CFG.AVENUE_GT}/*_label.mat'), key=natural_key)
av_labels = [np.array([int(np.asarray(loadmat(m)['volLabel'][0, i]).sum() > 0) for i in range(loadmat(m)['volLabel'].shape[1])]) for m in mats]
avis = sorted(glob.glob(f'{CFG.AVENUE_RAW}/testing_videos/*.avi'), key=natural_key)
av_frames = [cached_array(f'avenue_avi_{i+1:02d}', lambda p=p: frames_from_avi(p)) for i, p in enumerate(avis)]
train_frames = load_stack(list_frames(CFG.TRAIN_DIR, natural=False), 'avenue_train')
cross = {}
for name, (folder, lbl) in CFG.CROSS.items():
    cross[name] = dict(frames=load_stack(list_frames(folder, natural=False), 'cross_' + os.path.basename(folder)), label=lbl)
print(f'Avenue test {len(av_frames)} video | eğitim {len(train_frames)} kare | UCF {len(cross)} kategori')

In [ ]:
# A5 · UR Fall: 40 ADL (normal günlük aktivite) + 30 düşme videosu, kamera 0, RGB
def urf_sequence(name):
    cp, cn = f'{CFG.URF_CACHE}/{name}.npy', f'{CFG.URF_CACHE}/{name}_no.npy'
    if os.path.exists(cp): return np.load(cp), np.load(cn)
    z = f'{CFG.TMP}/{name}-cam0-rgb.zip'
    if not fetch(f'{CFG.URF_BASE}/{name}-cam0-rgb.zip', z): return None, None
    ex = f'{CFG.TMP}/{name}'
    with zipfile.ZipFile(z) as f: f.extractall(ex)
    pngs = sorted(glob.glob(f'{ex}/**/*.png', recursive=True), key=natural_key)
    arr = load_stack(pngs)
    no  = np.array([int(re.findall(r'(\d+)', os.path.basename(p))[-1]) for p in pngs])
    np.save(cp, arr); np.save(cn, no); shutil.rmtree(ex, ignore_errors=True); os.remove(z)
    return arr, no

urf = {}; fails = 0
t0 = time.time()
for kind, n in [('adl', CFG.N_ADL), ('fall', CFG.N_FALL)]:
    for i in range(1, n + 1):
        if fails >= 2 and not urf: break                      # sunucuya hiç ulaşılamıyor → UR Fall atlanır
        a, no = urf_sequence(f'{kind}-{i:02d}')
        if a is not None: urf[f'{kind}-{i:02d}'] = dict(frames=a, no=no, kind=kind)
        else: fails += 1
if not urf: print('⚠️ UR Fall indirilemedi; K1-a/K1-b atlanacak. Zip dosyalarını elle indirip CFG.TMP içine koyabilirsiniz.')
print(f"UR Fall: {sum(v['kind']=='adl' for v in urf.values())} ADL, {sum(v['kind']=='fall' for v in urf.values())} düşme videosu ({time.time()-t0:.0f} sn)")

# Düşme videolarının kare etiketleri: -1 = düşme öncesi, 0 = düşüyor, 1 = yerde
lab_csv = f'{CFG.URF_CACHE}/urfall-cam0-falls.csv'
if urf and not fetch(f'{CFG.URF_BASE}/urfall-cam0-falls.csv', lab_csv):
    urf = {k: v for k, v in urf.items() if v['kind'] == 'adl'}; print('⚠️ Düşme etiket dosyası indirilemedi')
fl = (pd.read_csv(lab_csv, header=None).iloc[:, :3] if os.path.exists(lab_csv) else pd.DataFrame(columns=[0, 1, 2])); fl.columns = ['seq', 'frame', 'label']
for k, v in urf.items():
    if v['kind'] == 'adl': v['flab'] = np.full(len(v['frames']), -1)
    else:
        d = dict(zip(fl[fl.seq == k].frame, fl[fl.seq == k].label))
        v['flab'] = np.array([d.get(int(x), -9) for x in v['no']])    # -9: etiketsiz kare

In [ ]:
# A6 · UR Fall pencereleri ve iki test kurulumu
#   K1-a  Alan içi:      ADL penceresi (normal)  vs  düşme/yerde penceresi (anomali)
#   K1-b  Aynı video içi: düşme videosunun olay öncesi penceresi  vs  aynı videonun olay penceresi
for k, v in urf.items():
    v['idx'] = win_idx(len(v['frames']), CFG.SEQ_STEP, CFG.URF_STRIDE)   # pencere uzunluğu modele göre yeniden kurulur
    wl = np.array([v['flab'][w] for w in v['idx']]) if len(v['idx']) else np.zeros((0, CFG.SEQ_LEN))
    v['valid'] = (wl != -9).all(1) if len(wl) else np.zeros(0, bool)
    v['event'] = (wl >= 0).any(1) if len(wl) else np.zeros(0, bool)

def urf_sets(scores):
    # scores: {video: pencere skorları} → iki kurulumun (y, s) dizileri
    a_y, a_s, b_y, b_s = [], [], [], []
    for k, v in urf.items():
        s, ok = scores[k], urf[k]['valid']
        if v['kind'] == 'adl':
            a_y += [0] * int(ok.sum()); a_s += list(s[ok])
        else:
            ev = v['event'] & ok; pre = (~v['event']) & ok
            a_y += [1] * int(ev.sum()); a_s += list(s[ev])
            b_y += [0] * int(pre.sum()) + [1] * int(ev.sum()); b_s += list(s[pre]) + list(s[ev])
    return (np.array(a_y), np.array(a_s)), (np.array(b_y), np.array(b_s))

if urf:
    (ya, _), (yb, _) = urf_sets({k: np.zeros(len(v['idx'])) for k, v in urf.items()})
    print(f'K1-a: {np.sum(ya==0)} ADL, {np.sum(ya==1)} düşme penceresi | K1-b: {np.sum(yb==0)} olay öncesi, {np.sum(yb==1)} olay penceresi')

## Değerlendirme
Pencere uzunluğu 16 olan model için tüm pencereler (doğrulama, Avenue, UCF, UR Fall) 16 kareyle yeniden kurulur. Doğrulama bölmesi 01 notebook'undaki ile aynıdır (seed 42).

In [ ]:
def set_windows(L):
    # Pencere uzunluğunu değiştirir ve L'ye bağlı tüm pencere listelerini yeniden kurar
    CFG.SEQ_LEN = L
    st = win_idx(len(train_frames), CFG.SEQ_STEP, 1); n = len(st); vn = max(1, int(n * CFG.VAL_FRAC))
    _, vs = torch.utils.data.random_split(range(n), [n - vn, vn], generator=torch.Generator().manual_seed(CFG.VAL_SPLIT_SEED))
    for v in cross.values(): v['idx'] = win_idx(len(v['frames']), CFG.SEQ_STEP, 1)
    for v in urf.values():
        v['idx'] = win_idx(len(v['frames']), CFG.SEQ_STEP, CFG.URF_STRIDE)
        wl = np.array([v['flab'][w] for w in v['idx']]) if len(v['idx']) else np.zeros((0, L))
        v['valid'] = (wl != -9).all(1) if len(wl) else np.zeros(0, bool)
        v['event'] = (wl >= 0).any(1) if len(wl) else np.zeros(0, bool)
    return [st[i] for i in vs.indices]

ROWS, SC = [], {}
for name, (path, desc) in CFG.MODELS.items():
    if not os.path.exists(path): print(f'— {name}: model yok, atlandı'); continue
    sp = f'{CFG.OUT_DIR}/{os.path.basename(path).replace(".pth", ".npz")}'
    model, skip, L = load_model(path); val_idx = set_windows(L)
    if os.path.exists(sp): S = np.load(sp, allow_pickle=True)['S'].item(); print(f'{name}: kayıtlı skorlar')
    else:
        t0 = time.time(); sc = lambda fr, idx: score_windows(model, fr, idx)
        S = {'val': sc(train_frames, val_idx),
             'avenue': [frame_level(f, CFG.AV_STRIDE, sc) for f in av_frames],
             'cross': {k: sc(v['frames'], v['idx']) for k, v in cross.items()},
             'urf': {k: (sc(v['frames'], v['idx']) if len(v['idx']) else np.zeros(0)) for k, v in urf.items()}}
        np.savez(sp, S=np.array(S, dtype=object)); print(f'{name}: skorlandı ({time.time()-t0:.0f} sn)')
    SC[name] = S
    y = np.concatenate(av_labels); sa = np.concatenate(S['avenue']); thr = np.percentile(S['val'], 95)
    ys = np.concatenate([np.full(len(S['cross'][k]), cross[k]['label']) for k in cross]); ss = np.concatenate([S['cross'][k] for k in cross])
    r = {'Varyant': name, 'Açıklama': desc, 'Parametre': sum(p.numel() for p in model.parameters()),
         'Normal doğrulama MSE (×1e-5)': round(1e5 * float(S['val'].mean()), 2),
         'Avenue mikro AUC %': round(100 * auc(y, sa), 2),
         'Avenue makro AUC %': round(100 * float(np.nanmean([auc(l, s) for l, s in zip(av_labels, S['avenue'])])), 2),
         'Avenue anomali/normal MSE oranı': round(float(sa[y == 1].mean() / sa[y == 0].mean()), 2),
         'Avenue FPR P95 %': round(100 * float((sa[y == 0] >= thr).mean()), 2),
         'Avenue→UCF-Crime AUC %': round(100 * auc(ys, ss), 2)}
    if urf:
        (ya, sa_), (yb, sb) = urf_sets(S['urf'])
        r['Avenue→UR Fall ADL–düşme AUC %'] = round(100 * auc(ya, sa_), 2)
        r['UR Fall aynı video içi AUC %'] = round(100 * auc(yb, sb), 2)
    ROWS.append(r); del model; torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
set_windows(8)
T_ABL = pd.DataFrame(ROWS); display(T_ABL)

In [ ]:
# Ablasyon şekli: her varyant, dört test kurulumunda
cols = [c for c in ['Avenue mikro AUC %', 'Avenue→UCF-Crime AUC %', 'Avenue→UR Fall ADL–düşme AUC %', 'UR Fall aynı video içi AUC %'] if c in T_ABL]
fig, ax = plt.subplots(figsize=(10, 4.8))
T_ABL.set_index('Varyant')[cols].plot.bar(ax=ax, width=.8, colormap='tab10')
ax.axhline(50, color='gray', ls=':', lw=1); ax.set_ylim(0, 105); ax.set_ylabel('AUC (%)'); ax.set_xlabel('')
ax.grid(axis='y', alpha=.3); ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False)
plt.xticks(rotation=15, ha='right', fontsize=9); plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/sekil_ablasyon.png', dpi=300); plt.show()

## Hız ölçümü (Ö9)
Makaledeki model, tek pencere (8 kare, 64×64) ve 32'lik parti ile ölçülür. Akış (streaming) kullanımında her yeni kare bir yeni pencere oluşturduğu için **pencere/sn ≈ işlenebilen kare/sn**. Ölçüm yalnızca modelin kendisini kapsar; kareyi okuma ve küçültme süresi dahil değildir (metinde belirtin).

In [ ]:
def bench(dev, batch, n_iter, threads=None):
    if threads: torch.set_num_threads(threads)
    m = AE3D().to(dev).eval(); m.load_state_dict(torch.load(CFG.MODELS['Tam (makaledeki model)'][0], map_location=dev))
    x = torch.rand(batch, 1, 8, 64, 64, device=dev)
    with torch.no_grad():
        for _ in range(5): m(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter): m(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / n_iter
    return {'Cihaz': (torch.cuda.get_device_name(0) if dev.type == 'cuda' else f'CPU ({torch.get_num_threads()} iş parçacığı)'),
            'Parti': batch, 'Gecikme (ms / parti)': round(1000 * dt, 2), 'Pencere / sn': round(batch / dt, 1)}

speed = []
if torch.cuda.is_available():
    speed += [bench(torch.device('cuda'), 1, 200), bench(torch.device('cuda'), 32, 50)]
speed += [bench(torch.device('cpu'), 1, 20), bench(torch.device('cpu'), 32, 5)]
T_SPEED = pd.DataFrame(speed); display(T_SPEED)

In [ ]:
T_ABL.to_csv(f'{CFG.OUT_DIR}/ablasyon.csv', index=False, encoding='utf-8-sig')
T_SPEED.to_csv(f'{CFG.OUT_DIR}/hiz.csv', index=False, encoding='utf-8-sig')
def _md(df):
    try: return df.to_markdown(index=False)
    except Exception: return df.to_string(index=False)
txt = ['# Ablasyon ve hız sonuçları (otomatik üretildi)', '',
       'Tüm varyantlar seed 42 ile eğitildi; eşik, Avenue kare skoru ve test kurulumları 02 notebook\'u ile aynı.', '',
       '## Ablasyon (Ö1, Ö5)', _md(T_ABL.drop(columns=['Açıklama'])), '',
       'Not: "Normal doğrulama MSE" düşük ve "anomali/normal MSE oranı" 1\'e yakınsa model girişi kopyalıyor demektir (özdeşlik kısayolu, Ö1).', '',
       '## Hız (Ö9)', _md(T_SPEED), '',
       'Not: Kare okuma ve 64×64 küçültme süresi dahil değildir.']
open(f'{CFG.OUT_DIR}/ablasyon_sayilari.md', 'w', encoding='utf-8').write('\n'.join(txt))
print('\n'.join(txt)); print('\n✅ Kaydedildi →', CFG.OUT_DIR)